# Modelado Inicial: Penetración de Internet en América Latina

**Dataset:** Personas usuarias de Internet por grupo etario, países seleccionados, 2016–2022  
**Fuente:** CEPALSTAT — Comisión Económica para América Latina y el Caribe  
**Objetivo:** Preparar el experimento de regresión, entrenar modelos base y evaluar su desempeño inicial.

## 1. Carga y Exploración del Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

df = pd.read_csv('../outputs/datos_transformados_semana2.csv')

print(f'Shape: {df.shape}')
print(df.head())
print('\nTipos de dato:')
print(df.dtypes)
print('\nNulos:', df.isnull().sum().sum())

## 2. Variables del Dataset

In [ ]:
print("Variable objetivo: porcentaje_internet")
print(df["porcentaje_internet"].describe().round(2))

print(f'\nPaíses ({df["pais"].nunique()}): {sorted(df["pais"].unique())}')
print(f'Años: {sorted(df["anio"].unique())}')

grupos = [c for c in df.columns if 'grupo' in c]
print(f'\nGrupos etarios (dummy): {grupos}')

## 3. Separación de Predictoras y Variable Objetivo

In [ ]:
y = df['porcentaje_internet'].copy()
X = df.drop('porcentaje_internet', axis=1).copy()

print(f'X: {X.shape}  |  y: {y.shape}')
print(f'Features: {list(X.columns)}')

## 4. Codificación de País

In [ ]:
# Label Encoding para 'pais'. Los modelos de árbol no asumen ordinalidad,
# por lo que la asignación alfabética no introduce sesgo en Random Forest ni Gradient Boosting.
# Para la regresión lineal es una simplificación aceptable en este baseline.
X_encoded = X.copy()
le_pais = LabelEncoder()
X_encoded['pais'] = le_pais.fit_transform(X_encoded['pais'])

mapeo = dict(zip(le_pais.classes_, le_pais.transform(le_pais.classes_)))
print('Mapeo de países:')
for pais, cod in sorted(mapeo.items(), key=lambda x: x[1]):
    print(f'  {cod}: {pais}')

## 5. División Temporal: Entrenamiento y Prueba

In [ ]:
year_counts = df.groupby('anio').size()
print('Registros por año:')
for anio, n in year_counts.items():
    print(f'  {anio}: {n:3d} ({n/len(df)*100:.1f}%)')

In [ ]:
# Split temporal: train 2016-2020, test 2021-2022.
# Un split aleatorio filtraría datos futuros al entrenamiento (fuga de información).
train_years = [2016, 2017, 2018, 2019, 2020]
test_years  = [2021, 2022]

train_mask = df['anio'].isin(train_years)
test_mask  = df['anio'].isin(test_years)

X_train = X_encoded[train_mask].reset_index(drop=True)
X_test  = X_encoded[test_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_test  = y[test_mask].reset_index(drop=True)

print(f'Entrenamiento: {len(X_train)} obs. ({len(X_train)/len(df)*100:.1f}%) — años {train_years}')
print(f'Prueba:        {len(X_test)} obs. ({len(X_test)/len(df)*100:.1f}%) — años {test_years}')

### Distribución de la Variable Objetivo

In [ ]:
print(f'y_train (2016-2020): media={y_train.mean():.1f}  std={y_train.std():.1f}  rango=[{y_train.min()}, {y_train.max()}]')
print(f'y_test  (2021-2022): media={y_test.mean():.1f}  std={y_test.std():.1f}  rango=[{y_test.min()}, {y_test.max()}]')
print(f'Diferencia de medias (test - train): {y_test.mean() - y_train.mean():.1f} pp')

In [ ]:
print('Resumen del experimento:')
print(f'  X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'  X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')
print(f'  Features: {list(X_train.columns)}')

## 6. Preparación de Features

`anio` y `years_since_2016` son perfectamente colineales (`years_since_2016 = anio - 2016`). Se elimina `anio` para evitar redundancia que infla los coeficientes en la regresión lineal.

In [ ]:
X_train_final = X_train.drop(columns=['anio'])
X_test_final  = X_test.drop(columns=['anio'])

print(f'X_train_final: {X_train_final.shape}')
print(f'X_test_final:  {X_test_final.shape}')
print(f'Features: {list(X_train_final.columns)}')

## 7. Modelos de Regresión Base

Se entrenan tres modelos sin ajuste de hiperparámetros para establecer un baseline. La optimización queda para la siguiente iteración.

### 7.1 Regresión Lineal

Modelo más simple posible: asume relación lineal entre cada feature y el porcentaje de internet. Limitación: la codificación ordinal de `pais` le asigna una gradiente artificial que el modelo interpreta de forma literal.

In [10]:
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()
model_lr.fit(X_train_final, y_train)
y_pred_lr = model_lr.predict(X_test_final)

print(f"Regresión Lineal entrenada")
print(f"Predicciones generadas: {y_pred_lr.shape}")
print(f"Rango predicciones: [{y_pred_lr.min():.2f}, {y_pred_lr.max():.2f}]")

Regresión Lineal entrenada


Predicciones generadas: (70,)
Rango predicciones: [33.58, 98.29]


### 7.2 Random Forest

Ensemble de árboles con bagging. La predicción final es el promedio de todos los árboles, lo que reduce el sobreajuste. No asume ordinalidad, por lo que la codificación de `pais` no le introduce sesgo.

In [11]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(random_state=42)
model_rf.fit(X_train_final, y_train)
y_pred_rf = model_rf.predict(X_test_final)

print(f"Random Forest entrenado")
print(f"Predicciones generadas: {y_pred_rf.shape}")
print(f"Rango predicciones: [{y_pred_rf.min():.2f}, {y_pred_rf.max():.2f}]")

Random Forest entrenado
Predicciones generadas: (70,)
Rango predicciones: [17.73, 93.38]


### 7.3 Gradient Boosting

Boosting secuencial: cada árbol corrige los residuales del anterior. Tercer punto de comparación para evaluar si el boosting supera al ensemble paralelo en este dataset.

In [12]:
from sklearn.ensemble import GradientBoostingRegressor

model_gb = GradientBoostingRegressor(random_state=42)
model_gb.fit(X_train_final, y_train)
y_pred_gb = model_gb.predict(X_test_final)

print(f"Gradient Boosting entrenado")
print(f"Predicciones generadas: {y_pred_gb.shape}")
print(f"Rango predicciones: [{y_pred_gb.min():.2f}, {y_pred_gb.max():.2f}]")

Gradient Boosting entrenado
Predicciones generadas: (70,)
Rango predicciones: [16.35, 98.74]


### Exportación de Predicciones

In [ ]:
import os

predicciones = pd.DataFrame({
    'y_real':    y_test.values,
    'y_pred_lr': y_pred_lr,
    'y_pred_rf': y_pred_rf,
    'y_pred_gb': y_pred_gb
})

os.makedirs('../outputs', exist_ok=True)
predicciones.to_csv('../outputs/predicciones_modelos_semana3.csv', index=False)
print(f'CSV exportado: {predicciones.shape[0]} filas, {predicciones.shape[1]} columnas')
print(predicciones.head())

## 8. Evaluación de Modelos

Se evalúan los tres modelos sobre el **conjunto de prueba (2021–2022)** usando tres métricas estándar para regresión. Esta es una **comparación de baseline**: ningún modelo ha sido optimizado. Los resultados establecen un punto de referencia para la siguiente iteración con ajuste de hiperparámetros.

### Significado de las métricas

| Métrica | Qué mide | Interpretación práctica | Unidad |
|---------|----------|------------------------|--------|
| **MAE** (Error Absoluto Medio) | Promedio de \|real − predicho\| | Error típico de predicción. Robusto a outliers. | pp |
| **RMSE** (Raíz del Error Cuadrático Medio) | Raíz de la media de (real − predicho)² | Penaliza errores grandes. RMSE >> MAE indica predicciones muy alejadas del real. | pp |
| **R²** (Coeficiente de determinación) | 1 − SS_res / SS_tot | Varianza explicada por el modelo. R²=1 es ajuste perfecto; R²=0 equivale a predecir siempre la media. | [0–1] |

Las tres métricas se calculan sobre observaciones que el modelo no vio durante el entrenamiento.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pred_df = pd.read_csv('../outputs/predicciones_modelos_semana3.csv')
y_real = pred_df['y_real']

modelos_eval = {
    'Regresion Lineal':  pred_df['y_pred_lr'],
    'Random Forest':     pred_df['y_pred_rf'],
    'Gradient Boosting': pred_df['y_pred_gb']
}

resultados = {}
rows = []
for nombre, y_pred in modelos_eval.items():
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    resultados[nombre] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    rows.append({'Modelo': nombre, 'MAE (pp)': round(mae, 2),
                 'RMSE (pp)': round(rmse, 2), 'R²': round(r2, 3)})

tabla = pd.DataFrame(rows).set_index('Modelo')
tabla

In [ ]:
# Análisis de sesgo y error máximo por modelo
print(f'Modelo con mejor desempeño inicial: {min(resultados, key=lambda n: resultados[n]["MAE"])}')
print()
print(f'{"Modelo":<22} {"Sesgo (pp)":>12} {"Std resid.":>11} {"Max err":>9}')
print('-' * 58)
for nombre, yp in modelos_eval.items():
    resid = y_real - yp
    flag = '  << mayor inestabilidad' if abs(resid).max() > 25 else ''
    print(f'{nombre:<22} {resid.mean():>12.1f} {resid.std():>11.1f} {abs(resid).max():>9.1f}{flag}')
print()
print('Sesgo positivo = el modelo subestima (predice menos que el valor real).')
print('Esperado: el test set tiene media +16 pp mayor que el train set.')

In [ ]:
import os
os.makedirs('../outputs/figures/semana3', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, (nombre, y_pred) in zip(axes, modelos_eval.items()):
    r2 = resultados[nombre]['R2']
    ax.scatter(y_real, y_pred, alpha=0.65, color='#4C78A8', edgecolors='white', s=55)
    lims = [5, 102]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Prediccion perfecta')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Valor real (%)')
    ax.set_ylabel('Valor predicho (%)')
    ax.set_title(f'{nombre}\nR\u00b2 = {r2:.3f}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Real vs. Predicho \u2014 Conjunto de prueba (2021\u20132022)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/semana3/comparacion_modelos.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
residuals_dict = {nombre: (y_real - yp).values for nombre, yp in modelos_eval.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot de residuales
bp = axes[0].boxplot(
    residuals_dict.values(),
    labels=residuals_dict.keys(),
    patch_artist=True,
    boxprops=dict(facecolor='#4C78A8', alpha=0.7)
)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5, label='Error cero')
axes[0].set_ylabel('Residual (real − predicho) en pp')
axes[0].set_title('Distribución de Residuales')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=15)

# Histograma superpuesto
colors = ['#4C78A8', '#F58518', '#54A24B']
for (nombre, res), color in zip(residuals_dict.items(), colors):
    axes[1].hist(res, bins=15, alpha=0.5, label=nombre, color=color, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5, label='Error cero')
axes[1].set_xlabel('Residual (pp)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Histograma de Residuales')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('Análisis de Errores — Conjunto de prueba (2021–2022)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/semana3/residuales.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
importancias = pd.DataFrame({
    'feature':           list(X_train_final.columns),
    'Random Forest':     model_rf.feature_importances_,
    'Gradient Boosting': model_gb.feature_importances_
}).set_index('feature').sort_values('Random Forest', ascending=False)

print('Importancia de variables:')
print(importancias.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ['Random Forest', 'Gradient Boosting']):
    df_plot = importancias[col].sort_values()
    ax.barh(df_plot.index, df_plot.values, color='#4C78A8', alpha=0.8)
    ax.set_xlabel('Importancia relativa')
    ax.set_title(f'Importancia de Variables \u2014 {col}')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/semana3/importancia_variables.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Conclusiones

### Comparación de modelos

| Modelo | MAE (pp) | RMSE (pp) | R² | Mejor en |
|--------|----------|-----------|-----|----------|
| Regresión Lineal | 8.35 | 10.60 | 0.732 | — |
| Random Forest | 5.81 | 6.92 | 0.886 | RMSE, R² |
| **Gradient Boosting** | **5.48** | **6.92** | **0.886** | MAE |

**Mejor modelo de baseline:** Gradient Boosting. Comparte RMSE y R² con Random Forest, pero obtiene el MAE más bajo (5.48 pp vs 5.81 pp). Significa que, en promedio, su predicción se aleja 5.5 puntos porcentuales del valor real, explicando el 88.6% de la varianza.

### Estabilidad y sesgo

Los tres modelos presentan un **sesgo sistemático de subestimación** (residuales medios entre +1 y +6 pp, es decir, el modelo predice menos de lo real). Esto es esperable: el conjunto de prueba tiene una media ~16 pp mayor que el de entrenamiento porque la adopción de internet siguió creciendo en 2021–2022. Ningún modelo captura completamente esa tendencia extrapolada.

La Regresión Lineal muestra la mayor inestabilidad: error máximo de 27.4 pp (vs ~16–19 pp en ensemble) y mayor dispersión de residuales (std=10.5 pp). No hay comportamiento catastrófico, pero es el modelo menos fiable.

### Variables más relevantes

En ambos modelos de ensemble, `pais` y `years_since_2016` concentran la mayor parte de la importancia relativa, lo que confirma que las diferencias geográficas y la tendencia temporal son los factores dominantes en la adopción de internet.

### Alcance de esta evaluación

Esta es una comparación de baseline sin optimización. Los próximos pasos incluyen ajuste de hiperparámetros, regularización en la regresión lineal y validación cruzada temporal.